# Small GPT-Style Language Model for Amharic
### University Assignment: From-Scratch NumPy Implementation

This notebook demonstrates a complete, transparent autoregressive language model for **Amharic (አማርኛ)** built using **pure Python and NumPy**.

**Core Architecture Components Demonstrated:**
1. Character-level Unicode-aware Amharic Tokenizer
2. Learned Token & Positional Embeddings
3. Causal Multi-Head Self-Attention with Triangular Masking
4. Pre-Layer Normalization & GELU Activations
5. Position-wise Feed-Forward Network (FFN)
6. Residual Connections & LM Vocabulary Projection Head
7. Autoregressive Text Generation (Greedy, Temperature, Top-k)

## 1. Setup and Imports
Ensure project modules are on the Python path.

In [ ]:
import sys
import os
import numpy as np

# Add repository root to path if running inside notebooks/
sys.path.append(os.path.abspath(".."))

from src.config import GPTConfig
from src.tokenizer import AmharicTokenizer
from src.model import GPTLanguageModel
from src.utils import load_checkpoint
from src.generate import generate

print("NumPy Version:", np.__version__)
print("Imports loaded successfully!")

## 2. Inspect the Custom Character-Level Amharic Tokenizer
We load the trained vocabulary and test token encoding and decoding.

In [ ]:
tokenizer = AmharicTokenizer(vocab_file="../checkpoints/vocab.json")
print(f"Total Vocabulary Size: {tokenizer.vocab_size} tokens")

# Demonstrate encoding & decoding
sample_phrase = "ሰላም ለኢትዮጵያ እና ለመላው ዓለም ይሁን!"
encoded_ids = tokenizer.encode(sample_phrase)
decoded_text = tokenizer.decode(encoded_ids)

print("\n--- Tokenizer Verification ---")
print("Input Text :", sample_phrase)
print("Token IDs  :", encoded_ids[:15], "...")
print("Decoded    :", decoded_text)
assert sample_phrase == decoded_text, "Encoding-decoding mismatch!"

## 3. Load Model and Inspect Configuration
We restore the checkpoint parameters, model architecture, and inspect parameter counts.

In [ ]:
model, config, _, meta = load_checkpoint("../checkpoints")

print("--- Model Configuration ---")
print(f"Vocabulary Size      : {config.vocab_size}")
print(f"Context Window (T)   : {config.block_size}")
print(f"Embedding Dim (d)    : {config.n_embd}")
print(f"Number of Heads (H)  : {config.n_head}")
print(f"Head Dimension (d_k) : {config.head_dim}")
print(f"Transformer Layers   : {config.n_layer}")
print(f"FFN Hidden Dimension : {config.ffn_dim}")
print(f"Total Parameters     : {model.num_parameters():,}")
print(f"Trained Steps        : {meta.get('step')}")
print(f"Final Val Loss       : {meta.get('val_loss'):.4f}")

## 4. Autoregressive Generation Comparison
We compare **Greedy Decoding**, **Low Temperature Sampling**, and **High Temperature with Top-K Filtering**.

In [ ]:
prompt = "ሰላም"

print("=" * 60)
print(f"PROMPT: '{prompt}'")
print("=" * 60)

# 1. Greedy Generation (Deterministic argmax)
output_greedy = generate(
    model=model,
    tokenizer=tokenizer,
    prompt=prompt,
    max_new_tokens=60,
    greedy=True
)
print("\n1. [Greedy Decoding]:")
print(output_greedy)

# 2. Low Temperature (T=0.6, Top-k=15)
output_temp_low = generate(
    model=model,
    tokenizer=tokenizer,
    prompt=prompt,
    max_new_tokens=60,
    temperature=0.6,
    top_k=15,
    greedy=False,
    rng=np.random.RandomState(42)
)
print("\n2. [Sampling with T=0.6, Top-k=15]:")
print(output_temp_low)

# 3. Higher Temperature (T=0.9, Top-k=20)
output_temp_high = generate(
    model=model,
    tokenizer=tokenizer,
    prompt=prompt,
    max_new_tokens=60,
    temperature=0.9,
    top_k=20,
    greedy=False,
    rng=np.random.RandomState(99)
)
print("\n3. [Sampling with T=0.9, Top-k=20]:")
print(output_temp_high)

## 5. Multiple Amharic Prompts Demonstration
Testing diverse starting prompts to see how the model continues text across different themes.

In [ ]:
prompts = [
    "ትምህርት ለ",
    "ኢትዮጵያ ",
    "የኮምፒውተር ",
    "እውቀት "
]

for p in prompts:
    gen = generate(
        model=model,
        tokenizer=tokenizer,
        prompt=p,
        max_new_tokens=50,
        temperature=0.75,
        top_k=15,
        greedy=False,
        rng=np.random.RandomState(hash(p) % 10000)
    )
    print(f"\nPrompt: '{p}'")
    print("Generated:", gen)
    print("-" * 50)